<a href="https://colab.research.google.com/github/Rakhayeva/multilingual-persuasion-detection/blob/main/04_baseline_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04: Baseline Models

This notebook trains and evaluates baseline models for persuasion detection.
We use classical ML models (TF-IDF + Logistic Regression / SVM) as baselines
before moving to transformer-based models in notebook 05.

## Experiments
- **Baseline:** Train on EN → Test on EN (in-language performance)
- **H1 (cross-lingual gap):** Train on EN → Test on KZ and RU proxy sets

## Why classical ML first?
- Fast to train — no GPU required
- Interpretable — compatible with SHAP LinearExplainer (notebook 06)
- Provides lower-bound performance for comparison with transformers

In [1]:
#Path
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/NU/SEDS/multilingual-persuasion-detection'

Mounted at /content/drive


In [2]:
# Libraries
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

In [3]:
# Load all splits
df_train   = pd.read_csv(f'{BASE}/data/processed/train.csv')
df_val     = pd.read_csv(f'{BASE}/data/processed/val.csv')
df_test_en = pd.read_csv(f'{BASE}/data/processed/test_en.csv')
df_test_kz = pd.read_csv(f'{BASE}/data/processed/test_kz.csv')
df_test_ru = pd.read_csv(f'{BASE}/data/processed/test_ru.csv')

print("Loaded:")
print(f"  Train:   {len(df_train)}")
print(f"  Val:     {len(df_val)}")
print(f"  Test EN: {len(df_test_en)}")
print(f"  Test KZ: {len(df_test_kz)} (proxy)")
print(f"  Test RU: {len(df_test_ru)} (proxy)")

Loaded:
  Train:   746
  Val:     160
  Test EN: 160
  Test KZ: 101 (proxy)
  Test RU: 103 (proxy)


## Model Training

In [4]:
# ── Build pipelines ───────────────────────────────────────────────────────────
# TF-IDF converts text to numerical features (word/bigram frequencies).
# class_weight='balanced' compensates for the 58/42 class imbalance.
# We train two classifiers to compare linear vs margin-based approaches.

def build_pipeline(model_type='LR'):
    """
    Builds a TF-IDF + classifier pipeline.
    model_type: 'LR' for Logistic Regression, 'SVM' for LinearSVC.
    """
    clf = (
        LogisticRegression(max_iter=1000, class_weight='balanced')
        if model_type == 'LR'
        else LinearSVC(max_iter=2000, class_weight='balanced')
    )
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            max_features=20000,  # top 20k most frequent features
            ngram_range=(1, 2),  # unigrams and bigrams
            sublinear_tf=True,   # log-scale TF to reduce impact of very frequent words
            min_df=2             # ignore terms appearing in fewer than 2 documents
        )),
        ('clf', clf)
    ])

In [5]:
# Train both models on English training set
X_train = df_train['text_clean'].fillna('')
y_train = df_train['has_persuasion']

pipelines = {}
for model_type in ['LR', 'SVM']:
    print(f"Training {model_type}...")
    pipe = build_pipeline(model_type)
    pipe.fit(X_train, y_train)
    pipelines[model_type] = pipe
    print(f"  Done.")

print("\nBoth models trained on EN training set.")

Training LR...
  Done.
Training SVM...
  Done.

Both models trained on EN training set.


## Evaluation

In [6]:
# Evaluation function
# We use F1-macro as the primary metric because:
# 1. It treats both classes equally regardless of their size
# 2. It is the standard metric for persuasion detection tasks (SemEval)
# 3. It penalizes models that simply predict the majority class

def evaluate(pipeline, X_test, y_test, label=''):
    """
    Evaluates a pipeline and returns F1-macro score.
    Prints classification report for detailed per-class performance.
    """
    y_pred = pipeline.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred,
                                 target_names=['Not manipulative',
                                               'Manipulative'],
                                 zero_division=0))
    return f1

# Run all experiments
results = []

test_sets = [
    (df_test_en, 'EN',        'in-language',       'Baseline'),
    (df_test_kz, 'KZ(proxy)', 'cross-lingual (H1)', 'H1'),
    (df_test_ru, 'RU(proxy)', 'cross-lingual (H1)', 'H1'),
]

for model_type, pipe in pipelines.items():
    for df_test, test_lang, exp_type, hypothesis in test_sets:
        X_test = df_test['text_clean'].fillna('')
        y_test = df_test['has_persuasion']
        label  = f"{model_type} | Train: EN → Test: {test_lang}"

        f1 = evaluate(pipe, X_test, y_test, label)

        results.append({
            'Model':      model_type,
            'Train':      'EN',
            'Test':       test_lang,
            'Type':       exp_type,
            'Hypothesis': hypothesis,
            'F1-macro':   round(f1, 3)
        })


  LR | Train: EN → Test: EN
                  precision    recall  f1-score   support

Not manipulative       0.67      0.83      0.74        66
    Manipulative       0.86      0.71      0.78        94

        accuracy                           0.76       160
       macro avg       0.76      0.77      0.76       160
    weighted avg       0.78      0.76      0.76       160


  LR | Train: EN → Test: KZ(proxy)
                  precision    recall  f1-score   support

Not manipulative       0.50      1.00      0.67        51
    Manipulative       0.00      0.00      0.00        50

        accuracy                           0.50       101
       macro avg       0.25      0.50      0.34       101
    weighted avg       0.25      0.50      0.34       101


  LR | Train: EN → Test: RU(proxy)
                  precision    recall  f1-score   support

Not manipulative       0.47      0.96      0.63        49
    Manipulative       0.00      0.00      0.00        54

        accuracy     

**EN to EN (in-language baseline):**
Both LR and SVM achieve F1-macro of 0.76, which is a reasonable
baseline for persuasion detection. The Manipulative class has higher
precision (0.86/0.83) but lower recall (0.71/0.76), meaning the model
is conservative: it correctly identifies manipulation when it predicts
it, but misses some manipulative paragraphs.

**EN to KZ and EN to RU (cross-lingual transfer, H1):**
Both models show degenerate prediction on Kazakh and Russian test sets.
Precision, recall, and F1 for the Manipulative class drop to 0.00,
meaning the models predict only "Not manipulative" for every single
text. This is the clearest possible confirmation of H1: models trained
on English persuasion features learn representations that do not
transfer to Kazakh or Russian at all. The cross-lingual gap is not
gradual but complete.

**LR vs SVM:**
Both classifiers produce identical results on KZ and RU test sets,
and nearly identical results on EN (0.761 vs 0.759). This suggests
the bottleneck is not the classifier choice but the feature
representation: TF-IDF word features learned from English text carry
no discriminative signal in Kazakh or Russian.

In [7]:
#  Results table
df_results = pd.DataFrame(results)
print("Baseline Model Results:")
print(df_results.to_string(index=False))

# Save results
df_results.to_csv(f'{BASE}/results/metrics_baseline.csv', index=False)
print("\nSaved: results/metrics_baseline.csv")

# Save pipelines for SHAP analysis in notebook 06
# We save only LR pipeline - SHAP LinearExplainer requires linear models
os.makedirs(f'{BASE}/results/models', exist_ok=True)
with open(f'{BASE}/results/models/pipeline_lr.pkl', 'wb') as f:
    pickle.dump(pipelines['LR'], f)
print("Saved: results/models/pipeline_lr.pkl (for SHAP in notebook 06)")

Baseline Model Results:
Model Train      Test               Type Hypothesis  F1-macro
   LR    EN        EN        in-language   Baseline     0.761
   LR    EN KZ(proxy) cross-lingual (H1)         H1     0.336
   LR    EN RU(proxy) cross-lingual (H1)         H1     0.313
  SVM    EN        EN        in-language   Baseline     0.759
  SVM    EN KZ(proxy) cross-lingual (H1)         H1     0.336
  SVM    EN RU(proxy) cross-lingual (H1)         H1     0.313

Saved: results/metrics_baseline.csv
Saved: results/models/pipeline_lr.pkl (for SHAP in notebook 06)


## Summary

| Model | Train | Test | F1-macro | Result |
|-------|-------|------|----------|--------|
| LR | EN | EN | 0.761 | Baseline |
| SVM | EN | EN | 0.759 | Baseline |
| LR | EN | KZ (proxy) | 0.336 | H1 confirmed |
| SVM | EN | KZ (proxy) | 0.336 | H1 confirmed |
| LR | EN | RU (proxy) | 0.313 | H1 confirmed |
| SVM | EN | RU (proxy) | 0.313 | H1 confirmed |

**Key finding:** Classical ML models trained on English persuasion
data fail completely on Kazakh and Russian texts, predicting only
the majority class. The F1-macro drops from 0.76 to 0.31-0.34,
a reduction of more than 50%. This confirms H1 and demonstrates
that English-trained TF-IDF features carry no transferable signal
to Cyrillic-script languages with different vocabulary and structure.

**Saved files:**
- `results/metrics_baseline.csv`: all experiment results
- `results/models/pipeline_lr.pkl`: LR pipeline for SHAP analysis

**Next:** [`05_transformer_experiments.ipynb`](https://colab.research.google.com/drive/1RIkYLhgjaIArdcXjqXvWMGxpaXA42R8h?usp=sharing): XLM-RoBERTa fine-tuning,
which uses multilingual representations that may partially bridge
the cross-lingual gap observed here.